## **FILTER LAGI** HANA

In [11]:
# import pandas as pd
# df = pd.read_csv('hasil_audit_label.csv')
# df

import pandas as pd
df_kamus = pd.read_csv('master_emoji_id.csv')
df = pd.read_csv('hasil_audit_label.csv')
df


,nomor_baris,file_sumber,tweet,label_saat_ini,label_seharusnya,tingkat_keyakinan,alasan,status
0,1,negatif,wah belom liat muka gue lagi murka hahahaha 😡😤,negatif,ambiguous,Rendah,"Error: ""Could not resolve authentication metho...",PERLU_REVIEW
1,2,negatif,Mungkin kurang piknik adrenalin 🙄. Mereka yang...,negatif,ambiguous,Rendah,"Error: ""Could not resolve authentication metho...",PERLU_REVIEW
2,3,negatif,Yg membuat lidahku... 👅 Gugup tak bergerak 😰,negatif,ambiguous,Rendah,"Error: ""Could not resolve authentication metho...",PERLU_REVIEW
3,4,negatif,Lho kok buru-buru mengatakan orang marah? 🙄 Ke...,negatif,ambiguous,Rendah,"Error: ""Could not resolve authentication metho...",PERLU_REVIEW
4,5,negatif,Apalagi yang jualan Spotify premium banyak ban...,negatif,ambiguous,Rendah,"Error: ""Could not resolve authentication metho...",PERLU_REVIEW
...,...,...,...,...,...,...,...,...
19311,4693,positif,Aduh mahal kali bang bisa buat beli Honda Vari...,positif,ambiguous,Rendah,"Error: ""Could not resolve authentication metho...",PERLU_REVIEW
19312,4694,positif,😮😮😮😮😮😮😮😮😮aooooooooooooooooooo<<oooooooooo,positif,ambiguous,Rendah,"Error: ""Could not resolve authentication metho...",PERLU_REVIEW
19313,4695,positif,Gunanya kater ap bg😂,positif,ambiguous,Rendah,"Error: ""Could not resolve authentication metho...",PERLU_REVIEW
19314,4696,positif,Gercep Gak Nih🙌,positif,ambiguous,Rendah,"Error: ""Could not resolve authentication metho...",PERLU_REVIEW


In [12]:
import pandas as pd
import emoji

# Ubah ke bentuk Python dictionary agar proses pencarian (lookup) super cepat O(1)
kamus_emoji = dict(zip(df_kamus['Emoji'], df_kamus['Special_Tag_ID']))

def map_and_filter_emoji(text):
    # Cari semua emoji yang ada di dalam teks
    emoji_list = emoji.emoji_list(text)
    
    # Kelompokkan emoji unik untuk divalidasi ke kamus
    for em in emoji_list:
        char_emoji = em['emoji']
        # JIKA EMOJI TIDAK DITEMUKAN DI KAMUS MASTER -> DROP BARIS
        if char_emoji not in kamus_emoji:
            return None # Penanda bahwa baris ini harus didrop
            
    # Jika semua emoji lolos validasi (atau teks tidak punya emoji sama sekali), lakukan pemetaan
    text_mapped = text
    for char_emoji, tag_id in kamus_emoji.items():
        if char_emoji in text_mapped:
            # Tambahkan spasi di sekitar tag agar tidak menempel dengan kata lain
            text_mapped = text_mapped.replace(char_emoji, f" {tag_id} ")
            
    return text_mapped

# Proses Cell 1: Terapkan ke Dataframe
print(f"Total dataset sebelum validasi emoji: {len(df)} baris")

kept_rows_cell1 = []
for idx, row in df.iterrows():
    text_processed = map_and_filter_emoji(str(row['tweet']))
    
    if text_processed is None:
        # Jalankan Kriteria: Jika emoji tidak ditemukan di kamus -> DROP
        print(f"[-] Cell 1 Drop (Emoji tak terdaftar): '{row['tweet']}'")
        continue
        
    # Update teks asli dengan teks yang sudah di-mapping special tag
    row['tweet'] = text_processed
    kept_rows_cell1.append(row)

# Hasil akhir Cell 1
df_cell1 = pd.DataFrame(kept_rows_cell1).reset_index(drop=True)
print(f"\nTotal dataset lolos Cell 1: {len(df_cell1)} baris")

Total dataset sebelum validasi emoji: 19316 baris

Total dataset lolos Cell 1: 19316 baris


In [13]:
import re

# 1. Definisikan Blacklist Kata Asing
ENGLISH_KEYWORDS = {
    'the', 'is', 'are', 'am', 'was', 'were', 'and', 'or', 'this', 'that', 'it',
    'i', 'you', 'he', 'she', 'they', 'we', 'my', 'your', 'for', 'to', 'in', 'on',
    'at', 'of', 'with', 'by', 'but', 'so', 'very', 'good', 'bad', 'app', 'download',
    'upload', 'error', 'login', 'logout', 'update', 'delete', 'cancel', 'account',
    'game', 'player', 'server', 'link', 'click', 'website', 'page', 'loading', 'best', 
    'make', 'up', 'happy', 'Happy', 'sad', 'angry', 'love', 'hate', 'fun', 'boring', 
    'cool', 'awesome', 'birthday', 'congratulations', 'congrats', 'win', 'lose', 'winner', 'loser', 'friend',
    'yes', 'no', 'ok', 'okay', 'please', 'thanks', 'thank', 'you', 'welcome', 'sorry'
}

MALAY_KEYWORDS = {
    'ni', 'tak', 'je', 'nak', 'ke', 'kat', 'korang', 'pula', 'seronok', 
    'comel', 'budak', 'gila', 'sia', 'macam', 'hang', 'tgk'
}

# 2. Fungsi Menyederhanakan Huruf Berulang (Mengabaikan Special Tag)
def reduce_elongated_words(text):
    words = text.split()
    clean_words = []
    for word in words:
        # Cek jika kata bermula dengan '<' dan berakhir dengan '>', berarti itu Special Tag dari master.csv
        if word.startswith('<') and word.endswith('>'):
            clean_words.append(word)
            continue
        reduced_word = re.sub(r'(.)\1{2,}', r'\1', word)
        clean_words.append(reduced_word)
    return " ".join(clean_words)

# 3. Fungsi Cek Noise (Panjang Teks & Rasio Karakter)
def is_pure_noise_cell2(text_with_tags):
    # Hilangkan semua spasi agar perhitungan densitas akurat
    text_no_space = text_with_tags.replace(" ", "")
    
    # Hitung berapa banyak Special Tag yang ada di kalimat ini
    # Kita cari pattern seperti <EMOJI_LAUGH> atau tag apapun yang berada di dalam kurung siku/sudut
    all_tags = re.findall(r'<[^>]+>', text_no_space)
    total_tags_count = len(all_tags)
    
    # Bersihkan string dari special tag untuk menghitung alfabet murni
    text_pure_char = re.sub(r'<[^>]+>', '', text_no_space)
    
    # FILTER 1: Jika panjang total (huruf asli + jumlah tag emoji) <= 3 -> DROP
    if (len(text_pure_char) + total_tags_count) <= 3:
        return True, "Panjang teks <= 3 karakter"
        
    if len(text_no_space) == 0:
        return True, "Kosong"
        
    # Hitung alfabet murni [a-zA-Z]
    huruf_alfabet = re.findall(r'[a-zA-Z]', text_pure_char)
    jumlah_huruf = len(huruf_alfabet)
    
    # Karakter uji = sisa huruf + angka + simbol + (1 tag dianggap bernilai 1 karakter)
    total_karakter_uji = len(text_pure_char) + total_tags_count
    
    # FILTER 2: Jika alfabet lebih sedikit atau sama dengan non-alfabet (angka/simbol/tag) -> DROP
    if jumlah_huruf <= (total_karakter_uji - jumlah_huruf):
        return True, f"Rasio angka/simbol/tag terlalu dominan ({jumlah_huruf}/{total_karakter_uji} huruf)"
        
    return False, None

# 4. Fungsi Strict Check Bahasa Asing
def contains_foreign_word_cell2(text_with_tags):
    # Bersihkan tanda baca, tetapi amankan huruf dan struktur tag <...>
    text_clean = re.sub(r'[^a-zA-Z<\s>]', ' ', text_with_tags).lower()
    text_clean = reduce_elongated_words(text_clean)
    
    words = text_clean.split()
    for word in words:
        if word in ENGLISH_KEYWORDS:
            return True, f"Inggris ('{word}')"
        if word in MALAY_KEYWORDS:
            return True, f"Malaysia ('{word}')"
            
    return False, None

# 5. Pipeline Utama Cell 2
def filter_pipeline_cell2(df_input):
    print(f"Total dataset awal Cell 2: {len(df_input)} baris\n")
    kept_rows_cell2 = []
    
    for idx, row in df_input.iterrows():
        text = str(row['tweet'])
        
        # Filter A: Cek Noise & Densitas Huruf vs Tag
        is_noise, reason_noise = is_pure_noise_cell2(text)
        if is_noise:
            print(f"[-] Cell 2 Drop ({reason_noise}): '{text}'")
            continue
            
        # Filter B: Cek Kata Asing
        has_foreign, reason_lang = contains_foreign_word_cell2(text)
        if has_foreign:
            print(f"[-] Cell 2 Drop (Bahasa {reason_lang}): '{text}'")
            continue
            
        kept_rows_cell2.append(row)
        
    df_final = pd.DataFrame(kept_rows_cell2).reset_index(drop=True)
    print(f"\nTotal dataset BERSIH AKHIR: {len(df_final)} baris")
    return df_final

# Jalankan Pipeline Lanjutan di Cell 2
df_bersih_final = filter_pipeline_cell2(df_cell1)

Total dataset awal Cell 2: 19316 baris

[-] Cell 2 Drop (Bahasa Inggris ('bad')): 'Mungkin kurang piknik adrenalin  <Wajah Memutar Mata> . Mereka yang skeptis tuh kek bad-debitor yang mau seluruh data debitur bank hilang  <Bom> , atau kek mahasiswa-pemalas yang ngarepin data nilai seluruh kampus hancur  <Tertawa Terpingkal-Pingkal> . Kaum "merasa" paling sengsara yang gak mo pusing sendirian oleh masalahnya  <Wajah Tidak Terhibur> .'
[-] Cell 2 Drop (Bahasa Malaysia ('tak')): 'Yg membuat lidahku...  <Lidah>  Gugup tak bergerak  <Wajah Mulut Terbuka Dan Keringat Dingin> '
[-] Cell 2 Drop (Bahasa Malaysia ('pula')): 'Iya lah, sekarang sudah kaya jadi mulai sombong pula. Cih, benci!  <Wajah Bersungut Marah>  <Wajah Tidak Terhibur> '
[-] Cell 2 Drop (Bahasa Malaysia ('ke')): 'Habis itu aku bilang gitu malah tanya tanya komitmen kek yang ada di foto ke 2  <Wajah Memutar Mata> . Kan katanya mau bilang langsung, ya emg dia besoknya jelasin langsung ke aku tp dg alasan yang kurang jelas  <Waja

In [15]:
df_bersih_final.to_csv('hasil_audit_label_filtering.csv', index=False) 
df_bersih_final

,nomor_baris,file_sumber,tweet,label_saat_ini,label_seharusnya,tingkat_keyakinan,alasan,status
0,1,negatif,wah belom liat muka gue lagi murka hahahaha <...,negatif,ambiguous,Rendah,"Error: ""Could not resolve authentication metho...",PERLU_REVIEW
1,4,negatif,Lho kok buru-buru mengatakan orang marah? <Wa...,negatif,ambiguous,Rendah,"Error: ""Could not resolve authentication metho...",PERLU_REVIEW
2,5,negatif,Apalagi yang jualan Spotify premium banyak ban...,negatif,ambiguous,Rendah,"Error: ""Could not resolve authentication metho...",PERLU_REVIEW
3,6,negatif,Buaya darat buset aku tertipu lagi <Wajah Men...,negatif,ambiguous,Rendah,"Error: ""Could not resolve authentication metho...",PERLU_REVIEW
4,7,negatif,"kalo lu gitugitu aja masih kurang syukur, gima...",negatif,ambiguous,Rendah,"Error: ""Could not resolve authentication metho...",PERLU_REVIEW
...,...,...,...,...,...,...,...,...
15206,4693,positif,Aduh mahal kali bang bisa buat beli Honda Vari...,positif,ambiguous,Rendah,"Error: ""Could not resolve authentication metho...",PERLU_REVIEW
15207,4694,positif,<Wajah Terperangah Terbuka> <Wajah Terperang...,positif,ambiguous,Rendah,"Error: ""Could not resolve authentication metho...",PERLU_REVIEW
15208,4695,positif,Gunanya kater ap bg <Wajah Menangis Gembira>,positif,ambiguous,Rendah,"Error: ""Could not resolve authentication metho...",PERLU_REVIEW
15209,4696,positif,Gercep Gak Nih <Mengangkat Kedua Tangan>,positif,ambiguous,Rendah,"Error: ""Could not resolve authentication metho...",PERLU_REVIEW
